# OLS

# OLS — Sale Price Models


## 1. Imports & settings


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, PolynomialFeatures, StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)


## 2. Load data (from processed CSV, work on a copy)


In [2]:
def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "processed" / "idealistaAPI").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto con data/processed/idealistaAPI")


def resolve_sale_csv_path(base_dir: Path) -> Path:
    explicit_csv = base_dir / "sale_homes_clean.csv"
    no_extension = base_dir / "sale_homes_clean"

    for path in [explicit_csv, no_extension]:
        if path.exists():
            return path

    matches = sorted(base_dir.glob("sale_homes_clean*"))
    if matches:
        return matches[0]

    raise FileNotFoundError("No se encontro un archivo compatible con sale_homes_clean en data/processed/idealistaAPI")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "idealistaAPI"
DATA_PATH = resolve_sale_csv_path(DATA_DIR)

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV detectado: {DATA_PATH.name}")
print(f"Shape original: {df_raw.shape}")
print("Primeras filas del CSV limpio de ventas:")
print(df_raw.head(3).to_string(index=False))


PROJECT_ROOT: /Users/sitomachucas/Documents/BezanillaSL
CSV detectado: sale_homes_clean.csv
Shape original: (604, 17)
Primeras filas del CSV limpio de ventas:
 codigo_inmueble   precio  superficie_construida_m2  numero_dormitorios  numero_banos tipologia      subtipologia provincia municipio                    distrito   latitud  longitud planta es_exterior tiene_ascensor tiene_garaje  obra_nueva
       110673583 375000.0                      79.0                   2             1      flat               NaN Cantabria Santander                   Valdenoja 43.480623 -3.797372      1        True           True         True       False
       110673566 500000.0                     223.0                   4             3    chalet semidetachedHouse Cantabria   Polanco                         NaN 43.387446 -4.017180    NaN         NaN            NaN         True       False
       110672622 190000.0                      56.0                   1             1      flat               NaN Cant

## 3. Cleaning & feature setup (explicit rules)


In [3]:
NUMERIC_CANDIDATES = [
    "precio",
    "superficie_construida_m2",
    "numero_dormitorios",
    "numero_banos",
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
]

BOOLEAN_CANDIDATES = [
    "es_exterior",
    "tiene_ascensor",
    "tiene_garaje",
    "obra_nueva",
]

CATEGORICAL_OHE_CANDIDATES = [
    "tipologia",
    "municipio",
]

IGNORED_COLUMNS = [
    "codigo_inmueble",
    "subtipologia",
    "distrito",
    "provincia",
]


def normalize_boolean_series(series: pd.Series) -> pd.Series:
    true_values = {"true", "t", "1", "yes", "y", "si", "s"}
    false_values = {"false", "f", "0", "no", "n"}

    def _convert(value):
        if pd.isna(value):
            return np.nan
        if isinstance(value, (bool, np.bool_)):
            return int(value)
        if isinstance(value, (int, np.integer, float, np.floating)) and value in (0, 1):
            return int(value)

        text = str(value).strip().lower()
        if text in true_values:
            return 1
        if text in false_values:
            return 0
        return np.nan

    return series.map(_convert).astype(float)


def safe_log1p(values):
    values = np.asarray(values, dtype=float)
    return np.log1p(np.clip(values, a_min=0.0, a_max=None))


def choose_reference_category(column_name: str, categories: list[str]) -> str:
    if column_name == "municipio" and "Santa Cruz de Bezana" in categories:
        return "Santa Cruz de Bezana"
    if column_name == "tipologia" and "flat" in categories:
        return "flat"
    if "missing" in categories:
        return "missing"
    return categories[0]


def build_category_maps(df_frame: pd.DataFrame, categorical_features: list[str]) -> tuple[dict, dict]:
    category_levels = {}
    drop_values = {}

    for column in categorical_features:
        filled_values = df_frame[column].fillna("missing").astype(str)
        categories = sorted(filled_values.unique().tolist())
        reference = choose_reference_category(column, categories)
        ordered_categories = [reference] + [value for value in categories if value != reference]
        category_levels[column] = ordered_categories
        drop_values[column] = reference

    return category_levels, drop_values


def make_one_hot_encoder(categories=None, drop=None):
    kwargs = {
        "handle_unknown": "ignore",
    }
    if categories is not None:
        kwargs["categories"] = categories
    if drop is not None:
        kwargs["drop"] = drop

    try:
        return OneHotEncoder(sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=False, **kwargs)


def build_generic_preprocessor(
    numeric_features,
    boolean_features,
    categorical_features,
    category_levels,
    drop_values,
    poly_features=None,
    log_features=None,
):
    poly_features = poly_features or []
    log_features = log_features or []
    transformers = []

    base_numeric = [feature for feature in numeric_features if feature not in poly_features]

    if base_numeric:
        transformers.append(
            (
                "num_base",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                base_numeric,
            )
        )

    if poly_features:
        transformers.append(
            (
                "num_poly",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                    ]
                ),
                poly_features,
            )
        )

    if log_features:
        transformers.append(
            (
                "num_log",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("log1p", FunctionTransformer(safe_log1p, feature_names_out="one-to-one")),
                    ]
                ),
                log_features,
            )
        )

    if boolean_features:
        transformers.append(
            (
                "bool",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                boolean_features,
            )
        )

    if categorical_features:
        ordered_categories = None
        dropped_categories = None

        if category_levels is not None:
            ordered_categories = [category_levels[column] for column in categorical_features]

        if drop_values is not None:
            dropped_categories = [drop_values[column] for column in categorical_features]

        transformers.append(
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                        (
                            "onehot",
                            make_one_hot_encoder(
                                categories=ordered_categories,
                                drop=dropped_categories,
                            ),
                        ),
                    ]
                ),
                categorical_features,
            )
        )

    return ColumnTransformer(transformers=transformers, remainder="drop")


def build_baseline_preprocessor(numeric_features, boolean_features, categorical_features, category_levels, drop_values):
    return build_generic_preprocessor(
        numeric_features=numeric_features,
        boolean_features=boolean_features,
        categorical_features=categorical_features,
        category_levels=category_levels,
        drop_values=drop_values,
    )


def build_enhanced_preprocessor(
    numeric_features,
    boolean_features,
    categorical_features,
    category_levels,
    drop_values,
    poly_features,
    log_features,
):
    return build_generic_preprocessor(
        numeric_features=numeric_features,
        boolean_features=boolean_features,
        categorical_features=categorical_features,
        category_levels=category_levels,
        drop_values=drop_values,
        poly_features=poly_features,
        log_features=log_features,
    )


def build_feature_selector(max_features=12):
    return SelectFromModel(
        estimator=LassoCV(cv=5, random_state=42, max_iter=10000),
        threshold=-np.inf,
        max_features=max_features,
    )


def compute_cv_mae(model_pipeline, X_train, y_train_raw, use_log_target, cv_splitter):
    fold_mae = []

    for fold_train_idx, fold_valid_idx in cv_splitter.split(X_train):
        X_fit = X_train.iloc[fold_train_idx].copy()
        X_valid = X_train.iloc[fold_valid_idx].copy()
        y_fit_raw = y_train_raw.iloc[fold_train_idx].copy()
        y_valid_raw = y_train_raw.iloc[fold_valid_idx].copy()
        y_fit = np.log1p(y_fit_raw) if use_log_target else y_fit_raw

        fold_model = clone(model_pipeline)
        fold_model.fit(X_fit, y_fit)

        fold_pred = fold_model.predict(X_valid)
        if use_log_target:
            fold_pred = np.expm1(fold_pred)

        fold_mae.append(mean_absolute_error(y_valid_raw, fold_pred))

    return float(np.mean(fold_mae)), float(np.std(fold_mae))


def get_final_feature_names(fitted_pipeline):
    feature_names = np.asarray(fitted_pipeline.named_steps["preprocessor"].get_feature_names_out(), dtype=object)

    if "selector" in fitted_pipeline.named_steps:
        feature_names = feature_names[fitted_pipeline.named_steps["selector"].get_support()]

    return feature_names.tolist()


def extract_design_matrix(fitted_pipeline, X_frame: pd.DataFrame):
    transformed = fitted_pipeline.named_steps["preprocessor"].transform(X_frame)
    feature_names = np.asarray(fitted_pipeline.named_steps["preprocessor"].get_feature_names_out(), dtype=object)

    if "scaler" in fitted_pipeline.named_steps:
        transformed = fitted_pipeline.named_steps["scaler"].transform(transformed)

    if "selector" in fitted_pipeline.named_steps:
        support_mask = fitted_pipeline.named_steps["selector"].get_support()
        transformed = fitted_pipeline.named_steps["selector"].transform(transformed)
        feature_names = feature_names[support_mask]

    transformed = np.asarray(transformed, dtype=float)
    return transformed, feature_names.tolist()


def build_statsmodels_artifacts(fitted_pipeline, X_train, y_train_raw, use_log_target):
    y_fit = np.log1p(y_train_raw) if use_log_target else y_train_raw
    X_design, feature_names = extract_design_matrix(fitted_pipeline, X_train)
    X_design_const = sm.add_constant(X_design, has_constant="add")

    sm_results = sm.OLS(y_fit, X_design_const).fit()

    coef_table = pd.DataFrame(
        {
            "term": ["const"] + feature_names,
            "coef": sm_results.params,
            "std_err": sm_results.bse,
            "t_value": sm_results.tvalues,
            "p_value": sm_results.pvalues,
        }
    )
    coef_table["abs_coef"] = coef_table["coef"].abs()

    const_row = coef_table[coef_table["term"] == "const"]
    other_rows = coef_table[coef_table["term"] != "const"].sort_values(
        ["abs_coef", "term"],
        ascending=[False, True],
    )
    coef_table = pd.concat([const_row, other_rows], ignore_index=True)

    for column in ["coef", "std_err", "t_value", "p_value", "abs_coef"]:
        coef_table[column] = coef_table[column].astype(float).round(6)

    return sm_results, coef_table


def evaluate_pipeline(model_name, model_pipeline, X_train, X_test, y_train_raw, y_test_raw, use_log_target, cv_splitter):
    cv_mae_mean, cv_mae_std = compute_cv_mae(
        model_pipeline=model_pipeline,
        X_train=X_train,
        y_train_raw=y_train_raw,
        use_log_target=use_log_target,
        cv_splitter=cv_splitter,
    )

    y_fit = np.log1p(y_train_raw) if use_log_target else y_train_raw
    fitted_pipeline = clone(model_pipeline)
    fitted_pipeline.fit(X_train, y_fit)

    y_pred = fitted_pipeline.predict(X_test)
    y_pred_eval = np.expm1(y_pred) if use_log_target else y_pred

    test_mae = mean_absolute_error(y_test_raw, y_pred_eval)
    test_rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_eval))
    test_r2 = r2_score(y_test_raw, y_pred_eval)

    sm_results, coef_table = build_statsmodels_artifacts(
        fitted_pipeline=fitted_pipeline,
        X_train=X_train,
        y_train_raw=y_train_raw,
        use_log_target=use_log_target,
    )

    result_row = {
        "model_name": model_name,
        "target": "precio",
        "transform": "log1p(target)" if use_log_target else "raw",
        "cv_MAE_mean": cv_mae_mean,
        "test_MAE": test_mae,
        "test_RMSE": test_rmse,
        "test_R2": test_r2,
        "n_features": len(get_final_feature_names(fitted_pipeline)),
    }

    print(model_name)
    print("Configuracion: train/test 80/20 + KFold=5 sobre train")
    print(f"CV MAE mean: {cv_mae_mean:,.2f} | CV MAE std: {cv_mae_std:,.2f}")
    print(f"TEST MAE: {test_mae:,.2f}")
    print(f"TEST RMSE: {test_rmse:,.2f}")
    print(f"TEST R2: {test_r2:.4f}")
    print(f"Numero de features finales: {result_row['n_features']}")
    print("Tabla de coeficientes (statsmodels, ordenada por |coef|):")
    print(coef_table.to_string(index=False))

    return result_row, fitted_pipeline, coef_table, sm_results


if "provincia" in df.columns:
    df = df.drop(columns=["provincia"])

if "precio" not in df.columns:
    raise ValueError("La columna 'precio' es obligatoria para entrenar las regresiones OLS")

# El target se mantiene no nulo y no negativo para soportar el caso log.
df = df[df["precio"].notna()].copy()
df = df[df["precio"] >= 0].copy()

for column in BOOLEAN_CANDIDATES:
    if column in df.columns:
        df[column] = normalize_boolean_series(df[column])

available_numeric_columns = [column for column in NUMERIC_CANDIDATES if column in df.columns]
numeric_feature_columns = [column for column in available_numeric_columns if column != "precio"]
available_boolean_columns = [column for column in BOOLEAN_CANDIDATES if column in df.columns]
available_categorical_columns = [column for column in CATEGORICAL_OHE_CANDIDATES if column in df.columns]
missing_numeric_columns = [column for column in NUMERIC_CANDIDATES if column not in df.columns]

feature_columns = numeric_feature_columns + available_boolean_columns + available_categorical_columns
if not feature_columns:
    raise ValueError("No hay features disponibles para entrenar los modelos")

X = df[feature_columns].copy()
y = df["precio"].copy()

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)
base_category_levels, base_drop_values = build_category_maps(X, available_categorical_columns)

log_feature_candidates = [
    "superficie_construida_m2",
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
]
poly_feature_candidates = [
    "superficie_construida_m2",
    "numero_dormitorios",
    "numero_banos",
]

log_feature_columns = [column for column in log_feature_candidates if column in numeric_feature_columns]
poly_feature_columns = [column for column in poly_feature_candidates if column in numeric_feature_columns]

results_rows = []
model_store = {}
coef_tables = {}
sm_results_store = {}

print(f"Shape de trabajo (copia): {df.shape}")
print(f"Columnas numericas reconocidas: {available_numeric_columns}")
print(f"Columnas numericas ausentes (se excluyen sin romper): {missing_numeric_columns}")
print(f"Features numericas finales: {numeric_feature_columns}")
print(f"Features booleanas finales: {available_boolean_columns}")
print(f"Features categoricas con OHE: {available_categorical_columns}")
print(f"Referencias OHE usadas en modelos base/mejorados: {base_drop_values}")
print(f"Columnas excluidas por regla: {[column for column in IGNORED_COLUMNS if column in df_raw.columns]}")
print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")
print("Todos los modelos del notebook usan train/test 80/20 y KFold=5 sobre train.")


Shape de trabajo (copia): (604, 16)
Columnas numericas reconocidas: ['precio', 'superficie_construida_m2', 'numero_dormitorios', 'numero_banos']
Columnas numericas ausentes (se excluyen sin romper): ['distancia_min_playa_km', 'distancia_min_supermercado_km', 'distancia_min_colegio_km']
Features numericas finales: ['superficie_construida_m2', 'numero_dormitorios', 'numero_banos']
Features booleanas finales: ['es_exterior', 'tiene_ascensor', 'tiene_garaje', 'obra_nueva']
Features categoricas con OHE: ['tipologia', 'municipio']
Referencias OHE usadas en modelos base/mejorados: {'tipologia': 'flat', 'municipio': 'Santa Cruz de Bezana'}
Columnas excluidas por regla: ['codigo_inmueble', 'subtipologia', 'distrito', 'provincia']
Train shape: (483, 9) | Test shape: (121, 9)
Todos los modelos del notebook usan train/test 80/20 y KFold=5 sobre train.


## 4. Baseline OLS (price)


In [4]:
baseline_pipeline_raw = Pipeline(
    steps=[
        (
            "preprocessor",
            build_baseline_preprocessor(
                numeric_features=numeric_feature_columns,
                boolean_features=available_boolean_columns,
                categorical_features=available_categorical_columns,
                category_levels=base_category_levels,
                drop_values=None,
            ),
        ),
        ("ols", LinearRegression()),
    ]
)

model_1_result, model_store["model_1"], coef_tables["model_1"], sm_results_store["model_1"] = evaluate_pipeline(
    model_name="Model 1 - OLS base",
    model_pipeline=baseline_pipeline_raw,
    X_train=X_train,
    X_test=X_test,
    y_train_raw=y_train_raw,
    y_test_raw=y_test_raw,
    use_log_target=False,
    cv_splitter=cv_splitter,
)
results_rows.append(model_1_result)


Model 1 - OLS base
Configuracion: train/test 80/20 + KFold=5 sobre train
CV MAE mean: 111,543.92 | CV MAE std: 14,535.34
TEST MAE: 139,581.82
TEST RMSE: 369,095.16
TEST R2: 0.2641
Numero de features finales: 69
Tabla de coeficientes (statsmodels, ordenada por |coef|):
                                 term           coef       std_err   t_value  p_value      abs_coef
                                const  -63227.339071  33527.460934 -1.885837 0.060007  63227.339071
                 cat__municipio_Getxo  558086.032590 131271.795423  4.251378 0.000026 558086.032590
     cat__municipio_Ribamontan al Mar  552038.356801 107461.360307  5.137087 0.000000 552038.356801
             cat__municipio_Castañeda -409082.451624 186109.047746 -2.198079 0.028488 409082.451624
             cat__municipio_Escalante  377853.646401 201920.004276  1.871304 0.061998 377853.646401
               cat__municipio_Limpias -313995.563356 183899.651132 -1.707429 0.088481 313995.563356
                cat__municipio_

## 5. Baseline OLS (log price)


In [5]:
baseline_pipeline_log = Pipeline(
    steps=[
        (
            "preprocessor",
            build_baseline_preprocessor(
                numeric_features=numeric_feature_columns,
                boolean_features=available_boolean_columns,
                categorical_features=available_categorical_columns,
                category_levels=base_category_levels,
                drop_values=None,
            ),
        ),
        ("ols", LinearRegression()),
    ]
)

model_2_result, model_store["model_2"], coef_tables["model_2"], sm_results_store["model_2"] = evaluate_pipeline(
    model_name="Model 2 - OLS base + log target",
    model_pipeline=baseline_pipeline_log,
    X_train=X_train,
    X_test=X_test,
    y_train_raw=y_train_raw,
    y_test_raw=y_test_raw,
    use_log_target=True,
    cv_splitter=cv_splitter,
)
results_rows.append(model_2_result)


Model 2 - OLS base + log target
Configuracion: train/test 80/20 + KFold=5 sobre train
CV MAE mean: 110,997.47 | CV MAE std: 22,583.08
TEST MAE: 125,084.19
TEST RMSE: 370,212.21
TEST R2: 0.2596
Numero de features finales: 69
Tabla de coeficientes (statsmodels, ordenada por |coef|):
                                 term      coef  std_err   t_value  p_value  abs_coef
                                const  4.781702 0.070637 67.694155 0.000000  4.781702
                   bool__tiene_garaje  4.781702 0.070637 67.694155 0.000000  4.781702
                cat__tipologia_chalet  1.772528 0.065928 26.885841 0.000000  1.772528
                  cat__tipologia_flat  1.749771 0.069891 25.035799 0.000000  1.749771
          cat__tipologia_countryHouse  1.259403 0.139047  9.057396 0.000000  1.259403
             cat__municipio_Escalante  1.165677 0.425412  2.740112 0.006404  1.165677
     cat__municipio_Ribamontan al Mar  1.127702 0.226403  4.980940 0.000001  1.127702
                 cat__municipi

## 6. Full OLS (all available regressors, dropped references) — log price

Este modelo fuerza un OLS con todas las variables disponibles del dataset de ventas, excepto `codigo_inmueble`, `subtipologia`, `distrito` y `provincia`.  
Tambien se excluye `precio` de los regresores porque es el target, y se fija `Santa Cruz de Bezana` como categoria de referencia para `municipio` para evitar multicolinealidad.


In [6]:
FORCED_DISTANCE_COLUMNS = [
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
]

FORCED_EXCLUDED_COLUMNS = {
    "codigo_inmueble",
    "subtipologia",
    "distrito",
    "provincia",
    "precio",
    "latitud",
    "longitud",
}

df_forced = df_raw.copy()

if "provincia" in df_forced.columns:
    df_forced = df_forced.drop(columns=["provincia"])

for column in FORCED_DISTANCE_COLUMNS:
    if column not in df_forced.columns:
        # El CSV de ventas no trae estas columnas; se crean en la copia para forzar el esquema completo sin tocar el original.
        df_forced[column] = 0.0

for column in BOOLEAN_CANDIDATES:
    if column in df_forced.columns:
        df_forced[column] = normalize_boolean_series(df_forced[column])

if "precio" not in df_forced.columns:
    raise ValueError("La columna 'precio' es obligatoria para el modelo OLS completo")

df_forced = df_forced[df_forced["precio"].notna()].copy()
df_forced = df_forced[df_forced["precio"] >= 0].copy()

forced_feature_columns = [
    column for column in df_forced.columns
    if column not in FORCED_EXCLUDED_COLUMNS
]

forced_boolean_features = [column for column in BOOLEAN_CANDIDATES if column in forced_feature_columns]
forced_categorical_features = []
forced_numeric_features = []

for column in forced_feature_columns:
    if column in forced_boolean_features:
        continue

    if pd.api.types.is_numeric_dtype(df_forced[column]):
        forced_numeric_features.append(column)
    else:
        forced_categorical_features.append(column)

forced_X = df_forced[forced_numeric_features + forced_boolean_features + forced_categorical_features].copy()
forced_y = df_forced["precio"].copy()

forced_X_train, forced_X_test, forced_y_train_raw, forced_y_test_raw = train_test_split(
    forced_X,
    forced_y,
    test_size=0.2,
    random_state=42,
)

forced_cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)
forced_category_levels, forced_drop_values = build_category_maps(forced_X_train, forced_categorical_features)

full_ols_log_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_baseline_preprocessor(
                numeric_features=forced_numeric_features,
                boolean_features=forced_boolean_features,
                categorical_features=forced_categorical_features,
                category_levels=forced_category_levels,
                drop_values=forced_drop_values,
            ),
        ),
        ("ols", LinearRegression()),
    ]
)

model_3_result, model_store["model_3"], coef_tables["model_3"], sm_results_store["model_3"] = evaluate_pipeline(
    model_name="Model 3 - OLS completo + log target",
    model_pipeline=full_ols_log_pipeline,
    X_train=forced_X_train,
    X_test=forced_X_test,
    y_train_raw=forced_y_train_raw,
    y_test_raw=forced_y_test_raw,
    use_log_target=True,
    cv_splitter=forced_cv_splitter,
)
results_rows.append(model_3_result)

print("Detalle del modelo 3:")
print(f"Features numericas forzadas: {forced_numeric_features}")
print(f"Features booleanas forzadas: {forced_boolean_features}")
print(f"Features categoricas forzadas: {forced_categorical_features}")
print(f"Referencias OHE del modelo 3: {forced_drop_values}")


Model 3 - OLS completo + log target
Configuracion: train/test 80/20 + KFold=5 sobre train
CV MAE mean: 115,429.05 | CV MAE std: 24,327.05
TEST MAE: 132,683.76
TEST RMSE: 375,857.44
TEST R2: 0.2368
Numero de features finales: 78
Tabla de coeficientes (statsmodels, ordenada por |coef|):
                                   term      coef  std_err   t_value  p_value  abs_coef
                                  const  5.711426 0.118835 48.061954 0.000000  5.711426
                     bool__tiene_garaje  5.711426 0.118835 48.061954 0.000000  5.711426
                 cat__municipio_Beranga -1.235477 0.315676 -3.913745 0.000106  1.235477
                 cat__municipio_Limpias -1.194084 0.410186 -2.911079 0.003799  1.194084
               cat__municipio_Castañeda -1.093062 0.412911 -2.647209 0.008430  1.093062
                          cat__planta_6  1.025071 0.199653  5.134258 0.000000  1.025071
               cat__municipio_Solorzano -1.020921 0.414680 -2.461947 0.014231  1.020921
    cat__m

/Users/sitomachucas/Documents/BezanillaSL/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## 7. Enhanced OLS (transformations + feature selection) — price

Se anaden `log1p` a `superficie_construida_m2` y a las distancias disponibles para reducir asimetria y hacer mas estable la relacion con el precio.  
Tambien se generan terminos polinomicos de grado 2 para `superficie_construida_m2`, `numero_dormitorios` y `numero_banos`, porque capturan curvaturas e interacciones sin meter complejidad innecesaria.

Para la seleccion de variables uso `SelectFromModel(LassoCV)` con un maximo de 12 features tras el preprocesado.  
Es una forma simple de quedarnos con las senales mas fuertes y contener el riesgo de sobreajuste antes de reentrenar el OLS final.


In [7]:
enhanced_pipeline_raw = Pipeline(
    steps=[
        (
            "preprocessor",
            build_enhanced_preprocessor(
                numeric_features=numeric_feature_columns,
                boolean_features=available_boolean_columns,
                categorical_features=available_categorical_columns,
                category_levels=base_category_levels,
                drop_values=base_drop_values,
                poly_features=poly_feature_columns,
                log_features=log_feature_columns,
            ),
        ),
        ("scaler", StandardScaler()),
        ("selector", build_feature_selector(max_features=12)),
        ("ols", LinearRegression()),
    ]
)

model_4_result, model_store["model_4"], coef_tables["model_4"], sm_results_store["model_4"] = evaluate_pipeline(
    model_name="Model 4 - OLS mejorado",
    model_pipeline=enhanced_pipeline_raw,
    X_train=X_train,
    X_test=X_test,
    y_train_raw=y_train_raw,
    y_test_raw=y_test_raw,
    use_log_target=False,
    cv_splitter=cv_splitter,
)
results_rows.append(model_4_result)


Model 4 - OLS mejorado
Configuracion: train/test 80/20 + KFold=5 sobre train
CV MAE mean: 124,167.86 | CV MAE std: 6,965.16
TEST MAE: 164,557.42
TEST RMSE: 391,206.28
TEST R2: 0.1732
Numero de features finales: 12
Tabla de coeficientes (statsmodels, ordenada por |coef|):
                                                 term           coef       std_err   t_value  p_value      abs_coef
                                                const  320514.060041   8564.916583 37.421737 0.000000 320514.060041
                             num_poly__numero_banos^2 -350920.086831  73393.366637 -4.781360 0.000002 350920.086831
num_poly__superficie_construida_m2 numero_dormitorios -304865.750040  69853.379825 -4.364366 0.000016 304865.750040
      num_poly__superficie_construida_m2 numero_banos  273192.362174  70664.368271  3.866055 0.000126 273192.362174
            num_poly__numero_dormitorios numero_banos  245146.318937 124052.846272  1.976144 0.048723 245146.318937
                    num_log__sup

## 8. Enhanced OLS (transformations + feature selection) — log price


In [8]:
enhanced_pipeline_log = Pipeline(
    steps=[
        (
            "preprocessor",
            build_enhanced_preprocessor(
                numeric_features=numeric_feature_columns,
                boolean_features=available_boolean_columns,
                categorical_features=available_categorical_columns,
                category_levels=base_category_levels,
                drop_values=base_drop_values,
                poly_features=poly_feature_columns,
                log_features=log_feature_columns,
            ),
        ),
        ("scaler", StandardScaler()),
        ("selector", build_feature_selector(max_features=12)),
        ("ols", LinearRegression()),
    ]
)

model_5_result, model_store["model_5"], coef_tables["model_5"], sm_results_store["model_5"] = evaluate_pipeline(
    model_name="Model 5 - OLS mejorado + log target",
    model_pipeline=enhanced_pipeline_log,
    X_train=X_train,
    X_test=X_test,
    y_train_raw=y_train_raw,
    y_test_raw=y_test_raw,
    use_log_target=True,
    cv_splitter=cv_splitter,
)
results_rows.append(model_5_result)


Model 5 - OLS mejorado + log target
Configuracion: train/test 80/20 + KFold=5 sobre train
CV MAE mean: 104,322.30 | CV MAE std: 13,991.95
TEST MAE: 142,769.02
TEST RMSE: 384,245.73
TEST R2: 0.2024
Numero de features finales: 12
Tabla de coeficientes (statsmodels, ordenada por |coef|):
                               term      coef  std_err    t_value  p_value  abs_coef
                              const 12.468178 0.018088 689.303062 0.000000 12.468178
  num_log__superficie_construida_m2  0.435937 0.033328  13.080220 0.000000  0.435937
             num_poly__numero_banos  0.194909 0.045026   4.328811 0.000018  0.194909
              cat__tipologia_chalet -0.152195 0.029665  -5.130371 0.000000  0.152195
           num_poly__numero_banos^2 -0.117867 0.039990  -2.947413 0.003364  0.117867
         cat__municipio_Torrelavega -0.116412 0.018688  -6.229274 0.000000  0.116412
               bool__tiene_ascensor  0.114514 0.020227   5.661321 0.000000  0.114514
        cat__tipologia_countryHous

## 9. Results summary table (5 models)


In [9]:
results_summary = pd.DataFrame(results_rows)[
    [
        "model_name",
        "target",
        "transform",
        "cv_MAE_mean",
        "test_MAE",
        "test_RMSE",
        "test_R2",
        "n_features",
    ]
].sort_values("test_MAE", ascending=True).reset_index(drop=True)

for column in ["cv_MAE_mean", "test_MAE", "test_RMSE"]:
    results_summary[column] = results_summary[column].round(2)
results_summary["test_R2"] = results_summary["test_R2"].round(4)

print("Resumen final (ordenado por test_MAE):")
print(results_summary.to_string(index=False))

print()
print("Tablas de coeficientes disponibles en memoria: coef_tables['model_1'] ... coef_tables['model_5']")


Resumen final (ordenado por test_MAE):
                         model_name target     transform  cv_MAE_mean  test_MAE  test_RMSE  test_R2  n_features
    Model 2 - OLS base + log target precio log1p(target)    110997.47 125084.19  370212.21   0.2596          69
Model 3 - OLS completo + log target precio log1p(target)    115429.05 132683.76  375857.44   0.2368          78
                 Model 1 - OLS base precio           raw    111543.92 139581.82  369095.16   0.2641          69
Model 5 - OLS mejorado + log target precio log1p(target)    104322.30 142769.02  384245.73   0.2024          12
             Model 4 - OLS mejorado precio           raw    124167.86 164557.42  391206.28   0.1732          12

Tablas de coeficientes disponibles en memoria: coef_tables['model_1'] ... coef_tables['model_5']
